In [71]:
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage,SystemMessage
from typing import TypedDict,Annotated
from pydantic import BaseModel,Field
from dotenv import load_dotenv
import operator
import os
import asyncio
from tavily import AsyncTavilyClient

In [72]:
load_dotenv()

True

In [73]:
model = ChatOpenAI(model='gpt-4o-mini')


In [74]:
class topic_analyser_schema(BaseModel):
    topic_summary : str = Field(description='Summary of the topic')
    web_research_questions :list[str] = Field(description='Web search question')
    statistics_questions :list[str] = Field(description='statistics questions')
    expert_opinion_questions :list[str] = Field(description='expert opinion questions')
    report_sections :list[str] = Field(description='report sections')

In [75]:
class research_synthesis_schema(BaseModel):
     question: list[str]
     findings: list[str]
     evidence: list[str]

In [76]:
structure_topic_analyser = model.with_structured_output(topic_analyser_schema)
structure_research_synthesis = model.with_structured_output(research_synthesis_schema)

In [77]:
class AgentState(TypedDict):
    topic: str
    topic_summary :str
    web_research_questions :list[str]
    statistics_questions :list[str]
    expert_opinion_questions :list[str]
    report_sections :list[str]
    web_research: list[object]
    statistics: list[object]
    expert_opinions: list[object]
    web_findings: list[dict]
    stats_findings: list[dict]
    expert_findings: list[dict]
    merged_research: list[dict]
    report: str
    quality_score: int
    iteration_count: int

In [78]:
def topic_analyser(state: AgentState):

    messages = [
        SystemMessage(
            content="""
You are an expert Research Planning Agent.

Your responsibility is to analyze a topic and create a structured research plan.

Do NOT perform research.

Generate:
- concise topic summary
- web research questions
- statistics research questions
- expert opinion questions
- report sections

Ensure all questions are:
- specific
- non-overlapping
- researchable
- relevant
"""
        ),
        HumanMessage(
            content=f"Topic: {state['topic']}"
        )
    ]

    response = structure_topic_analyser.invoke(messages)

    return response.model_dump()

In [79]:
client = AsyncTavilyClient(os.getenv('TAVILY_API_KEY'))

In [80]:
async def search_all(questions):
    # questions = state['web_research_questions']
    results = await asyncio.gather(*[client.search(q) for q in questions])
    return results


In [81]:
async def web_search(state:AgentState):
    results = await search_all(state['web_research_questions'])
    return {'web_research':results}

    

In [82]:
async def statistic_search(state:AgentState):
    results = await search_all(state['statistics_questions'])
    return {'statistics':results}


In [83]:
async def expert_opinion_questions(state:AgentState):
    results = await search_all(state['expert_opinion_questions'])
    return {'expert_opinions':results}


In [84]:
def research_synthesis(research_results, questions,role):
    prompt = f"""You are a {role}. Research Question: {questions} Search Results: {research_results}
               Analyze the search results and provide:
               1. Key Findings
               2. Supporting Evidence
               3. Important Trends
               4. Notable Examples
               Do not invent information.
               Use only the provided search results."""
    result = structure_research_synthesis.invoke(prompt)
    return [result]

In [85]:
def web_synthesis_node(state:AgentState):

    findings = research_synthesis(
        state["web_research"],
        state["web_research_questions"],
        "Web Research Analyst"
    )

    return {
        "web_findings": findings
    }

In [86]:
def statistic_synthesis_node(state:AgentState):

    findings = research_synthesis(
        state["statistics"],
        state["statistics_questions"],
        "Statistics Research Analyst"
    )
    return {
        "stats_findings": findings
    }

In [87]:
def expert_synthesis_node(state:AgentState):

    findings = research_synthesis(
        state["expert_opinions"],
        state["expert_opinion_questions"],
        "expert opinion Research Analyst"
    )
    return {
        "expert_findings": findings
    }

In [88]:
def merge_research(state: AgentState):

    web_search_findings = state["web_findings"]
    stats_findings = state["stats_findings"]
    expert_findings = state["expert_findings"]

    prompt = f"""
You are a Senior Research Analyst.

Your task is to merge research findings collected from multiple specialized research teams into a single unified research knowledge base.

Research Sources:

WEB RESEARCH FINDINGS:
{web_search_findings}

STATISTICAL FINDINGS:
{stats_findings}

EXPERT OPINION FINDINGS:
{expert_findings}

Instructions:

1. Review all findings carefully.
2. Remove duplicate information.
3. Combine related findings into cohesive insights.
4. Preserve important evidence, statistics, and expert viewpoints.
5. Organize findings into logical categories.
6. Highlight:
   - Key Trends
   - Industry Impact
   - Opportunities
   - Risks and Challenges
   - Statistical Evidence
   - Expert Insights
   - Future Outlook
7. Do NOT write a final report.
8. Do NOT create introductions or conclusions.
9. Focus on producing a structured research knowledge base that will later be used by a report generation agent.

Return the output in the following format:

{{
    "key_trends": [
        {{
            "insight": "...",
            "supporting_evidence": [...]
        }}
    ],

    "industry_impact": [
        {{
            "insight": "...",
            "supporting_evidence": [...]
        }}
    ],

    "opportunities": [
        {{
            "insight": "...",
            "supporting_evidence": [...]
        }}
    ],

    "risks_and_challenges": [
        {{
            "insight": "...",
            "supporting_evidence": [...]
        }}
    ],

    "statistical_evidence": [
        {{
            "insight": "...",
            "supporting_evidence": [...]
        }}
    ],

    "expert_insights": [
        {{
            "insight": "...",
            "supporting_evidence": [...]
        }}
    ],

    "future_outlook": [
        {{
            "insight": "...",
            "supporting_evidence": [...]
        }}
    ]
}}

Return only the structured result.
"""

    result = model.invoke(prompt)

    return {
        "merged_research": result.content
    }

In [89]:
graph = StateGraph(AgentState)
# Nodes
graph.add_node('topic_analyser',topic_analyser)
graph.add_node('web_search',web_search)
graph.add_node('statistic_search',statistic_search)
graph.add_node('expert_opinion_questions',expert_opinion_questions)
graph.add_node('web_synthesis_node',web_synthesis_node)
graph.add_node('statistic_synthesis_node',statistic_synthesis_node)
graph.add_node('expert_synthesis_node',expert_synthesis_node)
graph.add_node('merge_research',merge_research)

# Edges
graph.add_edge(START,'topic_analyser')
graph.add_edge('topic_analyser','web_search')
graph.add_edge('topic_analyser','statistic_search')
graph.add_edge('topic_analyser','expert_opinion_questions')
graph.add_edge('web_search','web_synthesis_node')
graph.add_edge('statistic_search','statistic_synthesis_node')
graph.add_edge('expert_opinion_questions','expert_synthesis_node')
graph.add_edge('web_synthesis_node','merge_research')
graph.add_edge('statistic_synthesis_node','merge_research')
graph.add_edge('expert_synthesis_node','merge_research')
graph.add_edge('merge_research',END)
workflow = graph.compile()

In [90]:
initialState = {
    'topic' : "Impact of Ai on software industry"
}
final_state =await workflow.ainvoke(initialState)
print (final_state)

{'topic': 'Impact of Ai on software industry', 'topic_summary': 'The impact of Artificial Intelligence (AI) on the software industry refers to how AI technologies are transforming software development, deployment, and maintenance processes. This includes automation of coding tasks, enhancement of software testing through predictive analytics, and the introduction of intelligent application features that improve user experiences. The software industry is leveraging AI for better decision-making, reduced time-to-market, and improved software quality, with implications for job roles, skills required, and overall industry structure.', 'web_research_questions': ['What are the key ways AI is being utilized in software development?', 'How has AI changed the software deployment process?', 'What are the recent trends in AI-driven software testing?', 'What is the market size of AI in the software industry?', 'Which companies are leading in the integration of AI in their software products?'], 'st